<a href="https://colab.research.google.com/github/mmatinm/SmartCane-AI-Visual-Navigation-System/blob/main/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Functions

In [3]:
import os
import yaml
import random
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

def report_yolo_dataset(dataset_dir, show_examples=True, num_examples=8):
    def count_images(split):
        img_dir = os.path.join(dataset_dir, split, "images")
        if os.path.exists(img_dir):
            return len([f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        else:
            return 0

    def analyze_split_labels(split, num_classes):
        label_dir = os.path.join(dataset_dir, split, "labels")
        class_image_counts = [0] * num_classes
        overlap_count = 0
        empty_or_missing_count = 0

        image_dir = os.path.join(dataset_dir, split, "images")
        if not os.path.exists(image_dir):
            return class_image_counts, overlap_count, empty_or_missing_count

        image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

        for img_file in image_files:
            img_stem = os.path.splitext(img_file)[0]
            label_path = os.path.join(label_dir, img_stem + ".txt")
            classes_in_image = set()

            if os.path.exists(label_path):
                with open(label_path, 'r') as f:
                    lines = f.readlines()
                    for line in lines:
                        parts = line.strip().split()
                        if len(parts) >= 1:
                            cls_id = int(float(parts[0]))
                            if 0 <= cls_id < num_classes:
                                classes_in_image.add(cls_id)
            else:
                empty_or_missing_count += 1
                continue  # no label file

            if len(classes_in_image) == 0:
                empty_or_missing_count += 1
            else:
                if len(classes_in_image) > 1:
                    overlap_count += 1
                for cls_id in classes_in_image:
                    class_image_counts[cls_id] += 1

        return class_image_counts, overlap_count, empty_or_missing_count

    # Print image counts
    train_count = count_images("train")
    valid_count = count_images("valid")
    test_count = count_images("test")

    print(f"📊 Image counts:")
    print(f"  Train: {train_count}")
    print(f"  Validation: {valid_count}")
    print(f"  Test: {test_count}")

    # Load class names
    data_yaml_path = os.path.join(dataset_dir, "data.yaml")
    with open(data_yaml_path, 'r') as f:
        data_yaml = yaml.safe_load(f)

    class_names = data_yaml.get("names", [])
    num_classes = len(class_names)

    print("\n🏷️ Classes in dataset:")
    for i, cls in enumerate(class_names):
        print(f"  {i}: {cls}")

    # Count per-class image occurrences and extra stats
    print("\n📂 Per-class image counts (number of images where each class appears):")
    for split in ["train", "valid", "test"]:
        class_image_counts, overlap_count, empty_count = analyze_split_labels(split, num_classes)
        print(f"  {split.capitalize()}:")
        for i, count in enumerate(class_image_counts):
            print(f"    {class_names[i]} ({i}): {count}")
        print(f"    ➕ Overlap images (contain ≥2 classes): {overlap_count}")
        print(f"    🚫 Images without any class (empty or missing labels): {empty_count}")

    if not show_examples:
        return

    # Pick examples from train split
    image_dir = os.path.join(dataset_dir, "train", "images")
    label_dir = os.path.join(dataset_dir, "train", "labels")

    image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if len(image_files) == 0:
        print("\n⚠️ No images found in train/images to display.")
        return

    sample_files = random.sample(image_files, min(num_examples, len(image_files)))

    print(f"\n🖼️ Showing {len(sample_files)} random annotated images...")

    fig, axes = plt.subplots(2, 4, figsize=(20, 10))
    axes = axes.flatten()

    for ax, fname in zip(axes, sample_files):
        img_path = os.path.join(image_dir, fname)
        label_path = os.path.join(label_dir, os.path.splitext(fname)[0] + ".txt")

        image = Image.open(img_path)
        w, h = image.size
        ax.imshow(image)
        ax.axis('off')

        if os.path.exists(label_path):
            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) != 5:
                        continue
                    cls_id, xc, yc, bw, bh = map(float, parts)
                    x = (xc - bw / 2) * w
                    y = (yc - bh / 2) * h
                    box_w = bw * w
                    box_h = bh * h
                    rect = patches.Rectangle((x, y), box_w, box_h,
                                             linewidth=2, edgecolor='red', facecolor='none')
                    ax.add_patch(rect)
                    if class_names:
                        ax.text(x, y, class_names[int(cls_id)], color='yellow', fontsize=10,
                                bbox=dict(facecolor='black', alpha=0.5))

    plt.tight_layout()
    plt.show()


# Merging

In [ ]:

!cp -r "/content/drive/MyDrive/SmartCane/final_bus" /content/
!cp -r "/content/drive/MyDrive/SmartCane/final_crosswalk" /content/
!cp -r "/content/drive/MyDrive/SmartCane/final_escalator" /content/
!cp -r "/content/drive/MyDrive/SmartCane/final_rglight" /content/
!cp -r "/content/drive/MyDrive/SmartCane/final_safety" /content/


In [ ]:
import os
import shutil
import yaml
from tqdm import tqdm

# List of dataset paths
datasets = [
    "/content/final_bus",
    "/content/final_crosswalk",
    "/content/final_escalator",
    "/content/final_rglight",
    "/content/final_safety"
]

# Output folder
output_dir = "/content/merged1"
splits = ["train", "valid", "test"]

# Make merged folders
for split in splits:
    os.makedirs(os.path.join(output_dir, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, split, "labels"), exist_ok=True)

global_class_names = []
class_offset = 0

# Merge process
for dataset in datasets:
    dataset_name = os.path.basename(dataset)

    # Read class names from data.yaml
    data_yaml_path = os.path.join(dataset, "data.yaml")
    with open(data_yaml_path, "r") as f:
        data_yaml = yaml.safe_load(f)

    local_names = data_yaml.get("names", [])
    if isinstance(local_names, dict):
        # convert dict to list
        local_names = [local_names[k] for k in sorted(local_names.keys())]

    # Append local names to global list
    global_class_names.extend(local_names)

    for split in splits:
        img_dir = os.path.join(dataset, split, "images")
        lbl_dir = os.path.join(dataset, split, "labels")
        out_img_dir = os.path.join(output_dir, split, "images")
        out_lbl_dir = os.path.join(output_dir, split, "labels")

        if not os.path.exists(img_dir):
            continue

        image_files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        pbar = tqdm(image_files, desc=f"Merging {split} from {dataset_name}", ncols=100)

        for img_file in pbar:
            base_name = os.path.splitext(img_file)[0]
            src_img_path = os.path.join(img_dir, img_file)
            src_lbl_path = os.path.join(lbl_dir, base_name + ".txt")

            # Rename with dataset prefix to avoid name clashes
            new_base = f"{dataset_name}_{base_name}"
            dst_img_path = os.path.join(out_img_dir, new_base + ".jpg")
            dst_lbl_path = os.path.join(out_lbl_dir, new_base + ".txt")

            shutil.copy2(src_img_path, dst_img_path)

            if os.path.exists(src_lbl_path):
                with open(src_lbl_path, "r") as f:
                    lines = f.readlines()

                new_lines = []
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    old_cls = int(parts[0])
                    new_cls = old_cls + class_offset
                    new_line = f"{new_cls} " + " ".join(parts[1:]) + "\n"
                    new_lines.append(new_line)

                with open(dst_lbl_path, "w") as f:
                    f.writelines(new_lines)

    class_offset += len(local_names)

# Write final merged data.yaml
final_yaml = {
    "train": os.path.join(output_dir, "train/images"),
    "val": os.path.join(output_dir, "valid/images"),
    "test": os.path.join(output_dir, "test/images"),
    "nc": len(global_class_names),
    "names": global_class_names
}

with open(os.path.join(output_dir, "data.yaml"), "w") as f:
    yaml.dump(final_yaml, f)

print("✅ Merge complete with class remapping.")


Merging test from final_safety: 100%|█████████████████████████████| 82/82 [00:00<00:00, 1439.70it/s]

✅ Merge complete with class remapping.


In [ ]:
report_yolo_dataset('/content/drive/MyDrive/SmartCane/final_dataset')

In [ ]:
import shutil

# Source and destination paths
src = "/content/merged1"
dst = "/content/drive/MyDrive/SmartCane/final_dataset"

# Copy folder recursively
shutil.copytree(src, dst)
print(f"✅ Dataset copied to {dst}")

✅ Dataset copied to /content/drive/MyDrive/SmartCane/final_dataset


# Preprocessing

In [4]:

!cp -r "/content/drive/MyDrive/SmartCane/final_dataset" /content/

In [ ]:
import os

# Define the folder and output zip file path
folder_path = "/content/drive/MyDrive/SmartCane/final_dataset"
zip_path = "/content/drive/MyDrive/SmartCane/final_dataset.zip"

# Run the zip command
!zip -r "{zip_path}" "{folder_path}"

In [5]:
import os
import cv2
import yaml
import shutil
import random
from glob import glob
from tqdm import tqdm
import albumentations as A

# Define augmentation pipeline
def get_augmentations():
    return A.Compose([
        A.Resize(640, 640),
        A.RandomBrightnessContrast(p=0.3),
        A.HorizontalFlip(p=0.5),
        A.Rotate(limit=15, p=0.4),
        A.Blur(p=0.2),
        A.CLAHE(p=0.1),
    ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.3))

# Load YOLO-format label
def load_yolo_label(label_path):
    boxes = []
    with open(label_path, 'r') as f:
        for line in f.readlines():
            parts = line.strip().split()
            if len(parts) >= 5:
                cls, x, y, w, h = map(float, parts[:5])
                boxes.append([x, y, w, h, int(cls)])
    return boxes

# Save YOLO-format label
def save_yolo_label(label_path, boxes):
    with open(label_path, 'w') as f:
        for box in boxes:
            x, y, w, h, cls = box
            f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")

# Augment and save
def augment_and_save(img_path, label_path, out_img_dir, out_label_dir, num_aug=3):
    image = cv2.imread(img_path)
    h, w = image.shape[:2]
    boxes = load_yolo_label(label_path)

    for i in range(num_aug):
        bboxes = [[x, y, bw, bh] for x, y, bw, bh, _ in boxes]
        class_labels = [cls for *_, cls in boxes]

        aug = get_augmentations()
        try:
            augmented = aug(image=image, bboxes=bboxes, class_labels=class_labels)
        except:
            continue

        new_img = augmented['image']
        new_boxes = augmented['bboxes']
        new_labels = augmented['class_labels']

        if not new_boxes:
            continue

        out_img_name = os.path.splitext(os.path.basename(img_path))[0] + f"_{i}.jpg"
        out_label_name = os.path.splitext(os.path.basename(label_path))[0] + f"_{i}.txt"

        cv2.imwrite(os.path.join(out_img_dir, out_img_name), new_img)
        save_yolo_label(os.path.join(out_label_dir, out_label_name),
                        [[*box, cls] for box, cls in zip(new_boxes, new_labels)])

# Preprocess dataset
def preprocess_yolo_dataset(input_dir, output_dir):
    splits = ['train', 'valid', 'test']
    for split in splits:
        print(f"Processing split: {split}")
        image_dir = os.path.join(input_dir, split, 'images')
        label_dir = os.path.join(input_dir, split, 'labels')

        out_img_dir = os.path.join(output_dir, split, 'images')
        out_label_dir = os.path.join(output_dir, split, 'labels')
        os.makedirs(out_img_dir, exist_ok=True)
        os.makedirs(out_label_dir, exist_ok=True)

        for img_path in tqdm(glob(os.path.join(image_dir, '*.jpg'))):
            base_name = os.path.splitext(os.path.basename(img_path))[0]
            label_path = os.path.join(label_dir, base_name + '.txt')

            if not os.path.exists(label_path):
                continue

            if split == 'train':
                # Apply 3 augmentations to each training image
                augment_and_save(img_path, label_path, out_img_dir, out_label_dir, num_aug=3)
            else:
                # For valid/test: just resize the image once without augmentation
                augment_and_save(img_path, label_path, out_img_dir, out_label_dir, num_aug=1)

# Generate data.yaml
def create_data_yaml(output_dir, class_names):
    data = {
        'train': os.path.join(output_dir, 'train', 'images'),
        'val': os.path.join(output_dir, 'valid', 'images'),
        'test': os.path.join(output_dir, 'test', 'images'),
        'nc': len(class_names),
        'names': class_names
    }

    with open(os.path.join(output_dir, 'data.yaml'), 'w') as f:
        yaml.dump(data, f)

def extract_class_names_from_yaml(yaml_path):
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)
    return data['names']

# Example usage

input_dataset = '/content/final_dataset'
output_dataset = '/content/augmented_dataset400'

# Dynamically extract class names
yaml_path = '/content/final_dataset/data.yaml'
class_names = extract_class_names_from_yaml(yaml_path)


preprocess_yolo_dataset(input_dataset, output_dataset)
create_data_yaml(output_dataset, class_names)


Processing split: train


100%|██████████| 7676/7676 [03:56<00:00, 32.44it/s]


Processing split: valid


100%|██████████| 1236/1236 [00:17<00:00, 68.92it/s]


Processing split: test


100%|██████████| 1482/1482 [00:19<00:00, 74.39it/s]


In [ ]:
import shutil

# Source and destination paths
src = "/content/augmented_dataset400"
dst = "/content/drive/MyDrive/SmartCane/augmented_dataset"

# Copy folder recursively
shutil.copytree(src, dst)
print(f"✅ Dataset copied to {dst}")

✅ Dataset copied to /content/drive/MyDrive/SmartCane/augmented_dataset


In [ ]:
import os
print(os.path.exists('/content/drive/MyDrive/SmartCane/augmented_dataset/data.yaml'))


True


In [ ]:
report_yolo_dataset('/content/final_dataset')

In [ ]:
import shutil

#shutil.rmtree('/content/augmented_dataset200')


# Train

In [ ]:

!cp -r "/content/drive/MyDrive/SmartCane/augmented_dataset" /content/

^C


In [6]:
import yaml
import os

yaml_path = "/content/augmented_dataset400/data.yaml"

# Load existing data.yaml
with open(yaml_path, 'r') as f:
    data = yaml.safe_load(f)

# Update paths to be relative
data['train'] = './train/images'
data['val'] = './valid/images'
data['test'] = './test/images'

# Save the updated YAML
with open(yaml_path, 'w') as f:
    yaml.dump(data, f, default_flow_style=False)

print("✅ data.yaml updated with relative paths.")


✅ data.yaml updated with relative paths.


In [ ]:
!pip install ultralytics


In [28]:
from ultralytics import YOLO

# Load model (you can change to yolov8m.pt or yolov8l.pt)
# Load pretrained model
model = YOLO("yolov8n.pt")



In [29]:
# Freeze all first
for param in model.model.parameters():
    param.requires_grad = False

# Unfreeze detection head + neck (from model.15 onward)
for i in [15, 16, 18, 19, 21, 22]:
    for param in model.model.model[i].parameters():
        param.requires_grad = True

In [ ]:
# Print layer trainability
frozen = []
trainable = []

for name, param in model.model.named_parameters():
    if not param.requires_grad:
        frozen.append(name)
    else:
        trainable.append(name)

print(f"🔒 Frozen layers: {len(frozen)}")
for name in frozen:
    print("  🔒", name)

print(f"\n🟢 Trainable layers: {len(trainable)}")
for name in trainable:
    print("  🟢", name)

In [10]:
!du -sh /content/augmented_dataset400


3.3G	/content/augmented_dataset400


In [32]:

model.train(
    data='/content/augmented_dataset400/data.yaml',  # absolute path to YAML
    epochs=10,
    imgsz=640,
    batch=8,
    device=0,  # first GPU
    name='yolov8_custom',
    project='/content/yolo_train_output',
    workers=4
)




Ultralytics 8.3.175 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/augmented_dataset400/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolov8_custom5, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=

train: Scanning /content/augmented_dataset400/train/labels.cache... 22515 images, 0 backgrounds, 0 corrupt: 100%|██████████| 22515/22515 [00:00<?, ?it/s]


albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 410.7±240.2 MB/s, size: 145.7 KB)


val: Scanning /content/augmented_dataset400/valid/labels.cache... 1211 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1211/1211 [00:00<?, ?it/s]


Plotting labels to /content/yolo_train_output/yolov8_custom5/labels.jpg... 
optimizer: 'optimizer=auto' found, ignoring 'lr0=0.01' and 'momentum=0.937' and determining best 'optimizer', 'lr0' and 'momentum' automatically... 
optimizer: AdamW(lr=0.000833, momentum=0.9) with parameter groups 57 weight(decay=0.0), 64 weight(decay=0.0005), 63 bias(decay=0.0)
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/yolo_train_output/yolov8_custom5
Starting training for 10 epochs...
Closing dataloader mosaic
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


       1/10       1.2G      1.611       3.12      1.851          3        640: 100%|██████████| 2815/2815 [08:16<00:00,  5.67it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 76/76 [00:13<00:00,  5.44it/s]

... (26 outputs left out)


10 epochs completed in 1.295 hours.
Optimizer stripped from /content/yolo_train_output/yolov8_custom5/weights/last.pt, 6.2MB
Optimizer stripped from /content/yolo_train_output/yolov8_custom5/weights/best.pt, 6.2MB

Validating /content/yolo_train_output/yolov8_custom5/weights/best.pt...
Ultralytics 8.3.175 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 76/76 [00:12<00:00,  6.28it/s]


                   all       1211       1379      0.844      0.779      0.837       0.53
               shelter        200        200      0.874      0.934      0.943      0.657
             crosswalk        200        205      0.921       0.91      0.964      0.799
             Escalator        177        299      0.763      0.699      0.738      0.426
             downstair         48         49      0.923      0.816      0.846      0.392
               upstair        102        112      0.724      0.563      0.663      0.321
                 green        200        210      0.818      0.824      0.869      0.498
                   red        200        208      0.865      0.774      0.877      0.533
           safety-line         84         96       0.86      0.708      0.792      0.617
Speed: 0.3ms preprocess, 2.5ms inference, 0.0ms loss, 2.1ms postprocess per image
Results saved to /content/yolo_train_output/yolov8_custom5


ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5, 6, 7])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7f4061fc0ed0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,

# save

In [35]:
# Define source and destination
src_path = "/content/yolo_train_output/yolov8_custom5/weights/best.pt"
dst_path = "/content/drive/MyDrive/SmartCane/yolov8n_0.83.pt"

# Copy the file
!cp "{src_path}" "{dst_path}"


In [ ]:
from ultralytics import YOLO

# Load the trained model from Google Drive
model = YOLO("/content/drive/MyDrive/SmartCane/best_yolov8_custom.pt")



# Test

In [33]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

def evaluate_yolov8_detector(model_path, data_yaml, split='test'):
    model = YOLO(model_path)

    # Run validation
    results = model.val(data=data_yaml, split=split, save_json=True, save_txt=True, iou=0.5)

    # Print key metrics
    print("\n📊 Detection Evaluation Metrics")
    print(f"mAP@0.5:       {results.metrics.map50:.3f}")
    print(f"mAP@0.5:0.95:  {results.metrics.map:.3f}")
    print(f"Precision:     {results.metrics.precision:.3f}")
    print(f"Recall:        {results.metrics.recall:.3f}")

    # Plot per-class results (optional)
    results.plot_metrics()  # This generates precision-recall curves, confusion, etc.

    # Visualize some predictions on test images
    test_imgs_dir = results.args.data['test']
    test_imgs = list(Path(test_imgs_dir).glob("*.jpg"))[:5]

    for img_path in test_imgs:
        im = cv2.imread(str(img_path))
        results = model.predict(source=im, conf=0.25, save=False)
        results[0].plot()  # overlay boxes
        plt.imshow(cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB))
        plt.title(f"Prediction: {img_path.name}")
        plt.axis('off')
        plt.show()


In [34]:
evaluate_yolov8_detector(
    model_path='/content/yolo_train_output/yolov8_custom5/weights/best.pt',
    data_yaml='/content/augmented_dataset400/data.yaml'
)


Ultralytics 8.3.175 🚀 Python-3.11.13 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 72 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 35.8±12.7 MB/s, size: 194.8 KB)


val: Scanning /content/augmented_dataset400/test/labels... 1455 images, 0 backgrounds, 0 corrupt: 100%|██████████| 1455/1455 [00:04<00:00, 328.61it/s]

val: New cache created: /content/augmented_dataset400/test/labels.cache



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 91/91 [00:25<00:00,  3.60it/s]


                   all       1455       1628      0.833      0.843      0.865      0.532
               shelter        200        200      0.891      0.955      0.972      0.685
             crosswalk        200        207      0.916      0.928       0.95        0.8
             Escalator        183        308      0.739      0.725      0.786      0.441
             downstair        199        200      0.892       0.89      0.921      0.405
               upstair        198        205      0.733      0.771      0.789      0.384
                 green        200        205      0.802      0.902      0.885      0.513
                   red        200        206      0.802       0.83      0.837      0.479
           safety-line         75         97      0.891      0.742      0.779      0.546
Speed: 0.5ms preprocess, 3.9ms inference, 0.0ms loss, 1.4ms postprocess per image
Saving runs/detect/val/predictions.json...
Results saved to runs/detect/val

📊 Detection Evaluation Metrics


AttributeError: 'DetMetrics' object has no attribute 'metrics'. See valid attributes below.

    Utility class for computing detection metrics such as precision, recall, and mean average precision (mAP).

    Attributes:
        names (Dict[int, str]): A dictionary of class names.
        box (Metric): An instance of the Metric class for storing detection results.
        speed (Dict[str, float]): A dictionary for storing execution times of different parts of the detection process.
        task (str): The task type, set to 'detect'.
        stats (Dict[str, List]): A dictionary containing lists for true positives, confidence scores, predicted classes, target classes, and target images.
        nt_per_class: Number of targets per class.
        nt_per_image: Number of targets per image.

    Methods:
        update_stats: Update statistics by appending new values to existing stat collections.
        process: Process predicted results for object detection and update metrics.
        cle